# Models
1. Math model, use "resaonable" assumption to get the result logically.
2. Pobability model, assign each "random varible" with  a pdf and numeric "variable" and get the result.
3. Machine Model (ML) model, use more variables, numeric, categorical, image etc data to get the result with (math) model without any logical rules: just get the result without understanding.

```
You can think that eagle can fly well without understanding aerodynamics.
```

## Procedure

```
1. Get data -> 2. data washing -> 3. data techniques -> 3. parameter tuning, Train-> 5. Predict
```

1. the most hard part
2. routine work, remove empty data or replace a suitable value
3. the most important part: data converting ( from non-numeric to numeric), dig new feature, for example, Man + campus graduate is a new feature, others
4. split fined data into train/test parts,  use different parameter to train the model on the "tain part data" and valid on "test part data" to fit model
5. make a predict model by the final model in local box or on cloud.   
   



## models
Roughtly, introdue the following in simple example:
1. Superived Model, outcomes is "Yes/NO" or "o,1,...,9"
2. Non-supervied Model, outcomes is s real value
3. Deep Learning, including CNN. 


# OS info
1. HP laptop with Intel i7 CPU, Nvidia GPU, RTX 2060, on Manjaro Linux, 32G Ram
2. Python 3.14.2, Visual code ( not jupyter conda not on Google drive)


# Step 1: Environment Setup & Package Installation

In [ ]:
# import the necessary packages
# Python calculation
import numpy as np
# packages for Database 
import pandas as pd
# visuazation
import matplotlib.pyplot as plt
import seaborn as sns

# native powerful Python ML packages, scikit-learn\

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
#from sklearn.metrics import classification_report, roc_auc_score, roc_curve
#from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error,roc_auc_score

# Importing the enterprise tree-based standard
from catboost import CatBoostRegressor,CatBoostClassifier

# Visual configurations
sns.set_theme(style="whitegrid")
print("CatBoost and Scikit-Learn frameworks successfully initialized.")

In [ ]:
# Load raw customer churn data containing both string and numeric features
data_url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df_raw = pd.read_csv(data_url)

# Drop unique identifier column (irrelevant for modeling)
df_raw.drop(columns=['customerID'], inplace=True)

print(f"Dataset Shape: {df_raw.shape[0]} customers | {df_raw.shape[1]} metrics")
df_raw.head(3)

In [ ]:
# Dtype, numeric: int/floar, non-numeric: object
df_raw.info()

In [ ]:
# 1. Fix broken data types: Force convert TotalCharges to numeric, coerce spaces to NaN
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')

# Impute newly generated NaN entries with 0 (since tenure for these rows is 0)
df_raw['TotalCharges'].fillna(0, inplace=True)

# 2. Convert string target variable ('Yes'/'No') to numeric binary integers (1/0)
df_raw['Churn'] = df_raw['Churn'].map({'Yes': 1, 'No': 0})

# Segregate structural feature columns
X = df_raw.drop(columns=['Churn'])
y = df_raw['Churn']

# Define types for explicit handling
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = [col for col in X.columns if col not in numeric_features]

print(f"Numeric Feature Matrix Columns: {numeric_features}")
print(f"Categorical Feature Matrix Columns: {categorical_features}")

train_test_split
---
Split data -> 80% training data, 20% test data

StandardScaler()
---
Real value -> new value between (0,1)

then each feature is the same important  

In [ ]:
categorical_features

In [ ]:
# Isolate training split before transformation to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Construct an automated preprocessing pipeline using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

# Fit and execute conversion pipelines
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Extract generated feature names for transparency
encoded_feature_names = preprocessor.get_feature_names_out()
print(f"Encoded Feature Vector Dimensions: {X_train_encoded.shape[1]} (Expanded from {X_train.shape[1]})")

In [ ]:
preprocessor

In [ ]:
# Fit standard parametric model on manually converted numerical matrix
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_encoded, y_train)

# Predict probabilities for the positive churn class (Column 1)
y_prob_lr = lr_model.predict_proba(X_test_encoded)[:, 1]

In [ ]:
# Identify indices of non-numeric string columns relative to the original X dataframe
cat_indices = [X.columns.get_loc(col) for col in categorical_features]

# Initialize CatBoost using original raw text training sets!
cat_model = CatBoostClassifier(
    iterations=400,
    learning_rate=0.05,
    depth=5,
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

# Pass raw strings directly. CatBoost translates them on-the-fly.
cat_model.fit(
    X_train, y_train, 
    cat_features=cat_indices, 
    eval_set=(X_test, y_test), 
    early_stopping_rounds=30
)

y_prob_cat = cat_model.predict_proba(X_test)[:, 1]

Good or Bad
---
Use Metic to determine how good these models, 
There are several well-known Meterics for Classifier, precison, recall, fa, support, and AUC, betwwen 0 and 1; the last one, AUC,  is the famous one: execellent while it approaches 1 and bad while it is near 0.

Use the following image to visualize the result, AUC is the area of region under the graph.


In [ ]:
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_cat = roc_auc_score(y_test, y_prob_cat)

print("\n=== MACHINE LEARNING ARCHITECTURAL EVALUATION ===")
print(f"Logistic Regression (Manual Conversion Pipeline) ROC-AUC: {auc_lr:.4f}")
print(f"CatBoost Classifier (Native Categorical Strategy) ROC-AUC: {auc_cat:.4f}")

# Plotting the ROC Curves

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_cat, tpr_cat, _ = roc_curve(y_test, y_prob_cat)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {auc_lr:.3f})', color='darkorange', lw=2)
plt.plot(fpr_cat, tpr_cat, label=f'CatBoost Native (AUC = {auc_cat:.3f})', color='purple', lw=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('ROC Curve: Parametric Baseline vs. Native Tree Architectures', fontweight='bold')
plt.legend(loc="lower right")
plt.show()

# Fine Tuned CatboostClassifier
The most important part is shorten the learning rate from  0.05 to 0.03

Don't expect too much, to get great improvent is to dig out new feature, which is still not used now.

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, f1_score

# 1. Initialize an Advanced, Heavy-Duty CatBoost Configuration
optimized_catboost = CatBoostClassifier(
    iterations=1000,                # Allow more trees to capture subtle patterns
    learning_rate=0.03,             # Lower learning rate for steady, precise gradient updates
    depth=6,                        # Balanced depth to map high-order feature interactions
    l2_leaf_reg=5,                  # L2 Regularization to suppress overfitting on sparse text categories
    
    # CRITICAL: Automatically scale weights inversely proportional to class frequencies
    auto_class_weights='Balanced',  
    
    early_stopping_rounds=50,       # Force halt if validation AUC halts improvement for 50 trees
    eval_metric='AUC',
    random_seed=42,
    verbose=100
)

# 2. Fit the optimized pipeline on the raw mixed-type data split
optimized_catboost.fit(
    X_train, y_train,
    cat_features=cat_indices,
    eval_set=(X_test, y_test),
    use_best_model=True             # Roll back to the iteration with the absolute highest validation score
)

# 3. Predict continuous probabilities
y_prob_opt = optimized_catboost.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import roc_curve

# Calculate False Positive and True Positive Rates across all possible thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_prob_opt)

# Youden's J-Statistic: J = True Positive Rate – False Positive Rate
j_scores = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_threshold = thresholds[optimal_idx]

print(f"Mathematically Optimal Operational Threshold: {optimal_threshold:.4f}\n")

# Apply custom operational threshold to convert probabilities to classes
y_pred_default = (y_prob_opt >= 0.5).astype(int)
y_pred_optimized = (y_prob_opt >= optimal_threshold).astype(int)

print("--- PERFORMANCE WITH DEFAULT THRESHOLD (0.50) ---")
print(classification_report(y_test, y_pred_default))

print("--- PERFORMANCE WITH OPTIMIZED THRESHOLD ---")
print(classification_report(y_test, y_pred_optimized))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Calculate ROC curves for all three models
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_cat, tpr_cat, _ = roc_curve(y_test, y_prob_cat)
fpr_opt, tpr_opt, thresholds_opt = roc_curve(y_test, y_prob_opt)

# Calculate exact Area Under the Curve (AUC) metrics
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_cat = roc_auc_score(y_test, y_prob_cat)
auc_opt = roc_auc_score(y_test, y_prob_opt)

# Identify the coordinates of the mathematically optimal Youden's threshold on the optimized curve
optimal_idx = np.argmax(tpr_opt - fpr_opt)
opt_fpr_coord = fpr_opt[optimal_idx]
opt_tpr_coord = tpr_opt[optimal_idx]

# Plot configurations
plt.figure(figsize=(10, 7))

# Plot the curves
plt.plot(fpr_lr, tpr_lr, label=f'1. Logistic Regression (Baseline) (AUC = {auc_lr:.4f})', color='#b2bec3', linestyle='--', lw=2)
plt.plot(fpr_cat, tpr_cat, label=f'2. Vanilla CatBoost (Default) (AUC = {auc_cat:.4f})', color='#0984e3', linestyle=':', lw=2.5)
plt.plot(fpr_opt, tpr_opt, label=f'3. Optimized CatBoost (Balanced) (AUC = {auc_opt:.4f})', color='#6c5ce7', lw=3)

# Highlight the mathematically optimal operating threshold point
plt.scatter(opt_fpr_coord, opt_tpr_coord, color='#d63031', s=120, zorder=5, 
            label=f'Optimal Operating Threshold ({optimal_threshold:.3f})')

# Draw annotations showing the business trade-off improvement
plt.annotate(
    f'Maximized Sensitivity:\nTPR = {opt_tpr_coord:.2f}, FPR = {opt_fpr_coord:.2f}',
    xy=(opt_fpr_coord, opt_tpr_coord),
    xytext=(opt_fpr_coord + 0.12, opt_tpr_coord - 0.15),
    arrowprops=dict(facecolor='#d63031', shrink=0.08, width=1.5, headwidth=8),
    fontsize=10, fontweight='bold', bbox=dict(boxstyle="round,pad=0.4", fc="#ffeaa7", alpha=0.6)
)

# Plot the 50/50 baseline random guess line
plt.plot([0, 1], [0, 1], color='#2d3436', alpha=0.4, linestyle='-', label='Random Baseline (AUC = 0.5000)')

# Graph formatting for corporate presentation
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold', labelpad=10)
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12, fontweight='bold', labelpad=10)
plt.title('Enterprise System Evolution: ROC Curve Benchmarking', fontsize=14, fontweight='bold', pad=15)
plt.legend(loc="lower right", fontsize=10, frameon=True, shadow=True)
plt.tight_layout()
plt.show()

print("\n=== SYSTEM PERFORMANCE SUMMATION ===")
print(f"Total AUC Variance Recovered via Optimization: +{((auc_opt - auc_lr) * 100):.2f}% improvement over baseline.")

In [ ]:


# 1. Define a brand-new, hypothetical customer profile (Mixed Tabular Profile)
# This profile represents a risky user: Month-to-month contract, high monthly charges, low tenure
new_customer_raw = pd.DataFrame([{
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 3,                     # Short lifecycle stage
    'PhoneService': 'Yes',
    'MultipleLines': 'No',
    'InternetService': 'Fiber optic', # High-churn indicator service tier
    'OnlineSecurity': 'No',
    'OnlineBackup': 'No',
    'DeviceProtection': 'No',
    'TechSupport': 'No',
    'StreamingTV': 'Yes',
    'StreamingMovies': 'No',
    'Contract': 'Month-to-month',     # Flexible, high-risk contract structure
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 89.85,          # High financial burden
    'TotalCharges': 269.55
}])

print("=== INGESTED PRODUCTION INFERENCE PAYLOAD ===")
display(new_customer_raw)


In [ ]:

# --- MODEL 1: LOGISTIC REGRESSION PREDICTION ---
# Requires manual column pipeline transformation
new_customer_encoded = preprocessor.transform(new_customer_raw)
prob_lr = lr_model.predict_proba(new_customer_encoded)[0, 1]
pred_lr = "CHURN (1)" if prob_lr >= 0.5 else "RETAIN (0)"

# --- MODEL 2: VANILLA CATBOOST PREDICTION ---
# Uses raw string format, relies on default structural threshold (0.50)
prob_cat_vanilla = cat_model.predict_proba(new_customer_raw)[0, 1]
pred_cat_vanilla = "CHURN (1)" if prob_cat_vanilla >= 0.5 else "RETAIN (0)"

# --- MODEL 3: OPTIMIZED PRODUCTION CATBOOST PREDICTION ---
# Uses raw string format, applies tuned operational threshold (Youden's J-Stat)
prob_cat_opt = optimized_catboost.predict_proba(new_customer_raw)[0, 1]
pred_cat_opt = "CHURN (1)" if prob_cat_opt >= optimal_threshold else "RETAIN (0)"


# --- VISUAL PREDICTION COMPARISON DASHBOARD ---
print("\n" + "="*55)
print("             REAL-TIME PREDICTION RESULTS")
print("="*55)
print(f"{'ML Architecture Strategy':<30} | {'Probability':<12} | {'Action Taken'}")
print("-"*55)
print(f"{'1. Logistic Regression (Baseline)':<30} | {prob_lr:.2%}      | {pred_lr}")
print(f"{'2. Vanilla CatBoost (Default)':<30} | {prob_cat_vanilla:.2%}      | {pred_cat_vanilla}")
print(f"{'3. Optimized CatBoost (Balanced)':<30} | {prob_cat_opt:.2%}      | {pred_cat_opt} *[Threshold: {optimal_threshold:.2f}]")
print("="*55)

Without doubt, the tuned catboost model got the best result.

Exercise
---
1. Since 2016, tree-based model, xgboost, is known for its ability, better than math model. Use LLM to compare the three famous tree-based models, xgboost, lightgbm, and catboost.
2. Use lightgbm to consider this model again (by LLM).
3. Any question and comment.

At better, you can email back the file after compled before 7/14. Wait for your result.